# Navegar e verificar lineage

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="08_auditoria_lineage"


1. Escolha proposta, run, modelo+versão, snapshot ou campanha; paginação limita o grafo retornado.

In [ ]:
from src.workbench.lineage_review import inspect_graph
external_id=params.external_id or (str(params.proposal_id) if params.proposal_id else "")
if not external_id: raise ValueError("Informe external_id ou proposal_id")
evidence=inspect_graph(cfg,params.lineage_kind,external_id,params.model_version)
display(pd.DataFrame(evidence["nodes"]));display(pd.DataFrame(evidence["edges"]))


2. Grafo e verificações reais. VERIFIED exige presença/proveniência; MISSING/MISMATCH bloqueiam confiança; legado continua identificado.

In [ ]:
from IPython.display import HTML
from src.workbench.lineage_review import render_graph
display(HTML(render_graph(evidence)))
display(pd.DataFrame(evidence["checks"]))


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
